In [0]:
%fs ls /Volumes/workspace/default/gridpulse/Gridpulse/

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/README.md,README.md,1231,1788581837000
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/buildings_sample.json,buildings_sample.json,1842,1788581843000
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/checksums_sha256.txt,checksums_sha256.txt,1614,1788581842000
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/consumption_readings_sample.parquet,consumption_readings_sample.parquet,36322,1788581844000
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/data/,data/,0,1788582821665
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/data_dictionary.csv,data_dictionary.csv,4178,1788581838000
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/data_dictionary.md,data_dictionary.md,3497,1788581837000
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/dq_requirements.md,dq_requirements.md,2071,1788581837000
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/meter_reading_drop_01.json,meter_reading_drop_01.json,36596,1788581842000
dbfs:/Volumes/workspace/default/gridpulse/Gridpulse/meter_reading_drop_02.json,meter_reading_drop_02.json,36262,1788581843000


In [0]:
# GridPulse - Week 03 Data Exploration

BASE_PATH = "/Volumes/workspace/default/gridpulse/Gridpulse"

# Load sample datasets
meters = spark.read.csv(
    f"{BASE_PATH}/meters_sample.csv",
    header=True,
    inferSchema=True
)

buildings = spark.read.json(
    f"{BASE_PATH}/buildings_sample.json"
)

import pandas as pd

consumption_pd = pd.read_parquet(
    f"{BASE_PATH}/consumption_readings_sample.parquet"
)

# Convert nanosecond timestamps to microsecond precision
for col in consumption_pd.columns:
    if pd.api.types.is_datetime64_any_dtype(consumption_pd[col]):
        consumption_pd[col] = consumption_pd[col].astype("datetime64[us]")

consumption = spark.createDataFrame(consumption_pd)

print("Consumption readings loaded:", consumption.count())

print("Meters loaded:", meters.count())
print("Buildings loaded:", buildings.count())
print("Consumption readings loaded:", consumption.count())

Consumption readings loaded: 500
Meters loaded: 20
Buildings loaded: 5
Consumption readings loaded: 500


In [0]:
display(consumption)

source_record_id,reading_id,meter_id,reading_ts,energy_kwh,active_power_kw,voltage_v,current_a,power_factor,reading_quality_flag,source_system,producer_run_id
SRC-RDG-0001-00000,RDG-MTR0001-00000,MTR0001,2026-01-01T00:00:00.000Z,1.3574,5.404,412.9,8.226,0.9186,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00001,RDG-MTR0001-00001,MTR0001,2026-01-01T00:15:00.000Z,1.4064,5.662,422.95,8.181,0.9447,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00002,RDG-MTR0001-00002,MTR0001,2026-01-01T00:30:00.000Z,1.393,5.5018,415.06,8.032,0.9528,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00003,RDG-MTR0001-00003,MTR0001,2026-01-01T00:45:00.000Z,1.4129,5.5369,415.5,8.439,0.9116,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00004,RDG-MTR0001-00004,MTR0001,2026-01-01T01:00:00.000Z,1.4734,5.9136,416.55,8.649,0.9477,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00005,RDG-MTR0001-00005,MTR0001,2026-01-01T01:15:00.000Z,1.5344,6.182,416.12,9.25,0.9273,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00006,RDG-MTR0001-00006,MTR0001,2026-01-01T01:30:00.000Z,1.4541,5.8263,416.94,8.982,0.8982,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00007,RDG-MTR0001-00007,MTR0001,2026-01-01T01:45:00.000Z,1.559,6.249,414.49,9.165,0.9497,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00008,RDG-MTR0001-00008,MTR0001,2026-01-01T02:00:00.000Z,1.4971,5.9639,408.78,9.068,0.9289,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1
SRC-RDG-0001-00009,RDG-MTR0001-00009,MTR0001,2026-01-01T02:15:00.000Z,1.4678,5.8702,416.61,8.957,0.9083,ACTUAL,GRIDPULSE_SIM,P05-GRIDPULSE-SOURCE-V1


In [0]:
print("=== METERS SCHEMA ===")
meters.printSchema()

print("=== BUILDINGS SCHEMA ===")
buildings.printSchema()

print("=== CONSUMPTION SCHEMA ===")
consumption.printSchema()

=== METERS SCHEMA ===
root
 |-- source_record_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- meter_serial_no: string (nullable = true)
 |-- building_id: string (nullable = true)
 |-- tariff_plan_id: string (nullable = true)
 |-- meter_type: string (nullable = true)
 |-- capacity_kw: double (nullable = true)
 |-- meter_status: string (nullable = true)
 |-- voltage_class: string (nullable = true)
 |-- installed_date: date (nullable = true)
 |-- effective_from: timestamp (nullable = true)
 |-- effective_to: timestamp (nullable = true)

=== BUILDINGS SCHEMA ===
root
 |-- _corrupt_record: string (nullable = true)

=== CONSUMPTION SCHEMA ===
root
 |-- source_record_id: string (nullable = true)
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- active_power_kw: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- current_a: dou

In [0]:
buildings_raw = spark.read.text(
    f"{BASE_PATH}/buildings_sample.json"
)

display(buildings_raw)

value
[
{
"""source_record_id"": ""SRC-BLD-001"","
"""building_id"": ""BLD001"","
"""building_name"": ""GridPulse Academic Block 01"","
"""building_type"": ""Academic"","
"""campus_zone"": ""North"","
"""floor_area_sqm"": 3450.0,"
"""operating_profile"": ""Weekday 07:30-18:30"","
"""criticality_band"": ""Medium"","


In [0]:
buildings = spark.read.option(
    "multiLine", "true"
).json(
    f"{BASE_PATH}/buildings_sample.json"
)

print("Buildings loaded:", buildings.count())

Buildings loaded: 5


In [0]:
meters.createOrReplaceTempView("meters")
buildings.createOrReplaceTempView("buildings")
consumption.createOrReplaceTempView("consumption")

print("SQL views created successfully")

SQL views created successfully


In [0]:
print("Meters:", meters.count())
print("Buildings:", buildings.count())
print("Consumption:", consumption.count())

Meters: 20
Buildings: 5
Consumption: 500


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

print("=== METERS NULL COUNTS ===")
display(
    meters.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in meters.columns
    ])
)

print("=== BUILDINGS NULL COUNTS ===")
display(
    buildings.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in buildings.columns
    ])
)

print("=== CONSUMPTION NULL COUNTS ===")
display(
    consumption.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in consumption.columns
    ])
)

=== METERS NULL COUNTS ===


source_record_id,meter_id,meter_serial_no,building_id,tariff_plan_id,meter_type,capacity_kw,meter_status,voltage_class,installed_date,effective_from,effective_to
0,0,0,0,0,0,0,0,0,0,0,0


=== BUILDINGS NULL COUNTS ===


active_flag,building_id,building_name,building_type,campus_zone,commissioning_date,criticality_band,floor_area_sqm,operating_profile,source_record_id
0,0,0,0,0,0,0,0,0,0


=== CONSUMPTION NULL COUNTS ===


source_record_id,reading_id,meter_id,reading_ts,energy_kwh,active_power_kw,voltage_v,current_a,power_factor,reading_quality_flag,source_system,producer_run_id
0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
negative_energy = consumption.filter(
    col("energy_kwh") < 0
)

print("Negative energy records:", negative_energy.count())

display(negative_energy)

Negative energy records: 0


source_record_id,reading_id,meter_id,reading_ts,energy_kwh,active_power_kw,voltage_v,current_a,power_factor,reading_quality_flag,source_system,producer_run_id


In [0]:
from pyspark.sql.functions import current_timestamp, col

future_readings = consumption.filter(
    col("reading_ts") > current_timestamp()
)

print("Future timestamp records:", future_readings.count())

display(future_readings)

Future timestamp records: 0


source_record_id,reading_id,meter_id,reading_ts,energy_kwh,active_power_kw,voltage_v,current_a,power_factor,reading_quality_flag,source_system,producer_run_id


In [0]:
missing_meters = consumption.join(
    meters.select("meter_id").distinct(),
    on="meter_id",
    how="left_anti"
)

print("Consumption records with missing meter IDs:", missing_meters.count())

display(missing_meters)

Consumption records with missing meter IDs: 0


meter_id,source_record_id,reading_id,reading_ts,energy_kwh,active_power_kw,voltage_v,current_a,power_factor,reading_quality_flag,source_system,producer_run_id


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

window_spec = Window.partitionBy("meter_id").orderBy("reading_ts")

out_of_order = (
    consumption
    .withColumn("previous_reading_ts", lag("reading_ts").over(window_spec))
    .filter(
        col("previous_reading_ts").isNotNull() &
        (col("reading_ts") < col("previous_reading_ts"))
    )
)

print("Out-of-order readings:", out_of_order.count())

display(out_of_order)

Out-of-order readings: 0


source_record_id,reading_id,meter_id,reading_ts,energy_kwh,active_power_kw,voltage_v,current_a,power_factor,reading_quality_flag,source_system,producer_run_id,previous_reading_ts
